In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

DATA_PATH = Path("../data/diabetic_data.csv")

df = pd.read_csv(DATA_PATH)

df.shape

(101766, 50)

In [3]:
df.head()


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [5]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [6]:
df["readmitted"].value_counts(normalize=True) * 100

readmitted
NO     53.911916
>30    34.928169
<30    11.159916
Name: proportion, dtype: float64

In [7]:
df["readmitted_30"] = np.where(df["readmitted"] == "<30", 1, 0)

df["readmitted_30"].value_counts()


readmitted_30
0    90409
1    11357
Name: count, dtype: int64

In [8]:
df["readmitted_30"].value_counts(normalize=True) * 100


readmitted_30
0    88.840084
1    11.159916
Name: proportion, dtype: float64

## 1.2 Análise da variável-alvo

A variável original `readmitted` apresenta três categorias: pacientes não readmitidos (`NO`), pacientes readmitidos após mais de 30 dias (`>30`) e pacientes readmitidos em menos de 30 dias (`<30`).

Para este projeto, foi criada uma variável binária denominada `readmitted_30`, em que a classe positiva corresponde aos pacientes readmitidos em menos de 30 dias. Esta decisão permite concentrar a análise no caso clinicamente mais crítico.

In [9]:
# Substituir "?" por valores em falta reais
df = df.replace("?", np.nan)

# Tabela de valores em falta
missing_table = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": df.isna().mean() * 100
}).sort_values("pct_missing", ascending=False)

missing_table[missing_table["n_missing"] > 0]

,n_missing,pct_missing
weight,98569,96.858479
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
medical_specialty,49949,49.082208
payer_code,40256,39.557416
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636


In [10]:
# Cópia do dataset para preparação
df_clean = df.copy()

# Colunas removidas por elevada percentagem de valores em falta
cols_to_drop = [
    "weight",
    "payer_code",
    "medical_specialty"
]

df_clean = df_clean.drop(columns=cols_to_drop)

df_clean.shape


(101766, 48)

In [11]:
# Preencher valores em falta em variáveis categóricas relevantes
categorical_missing_cols = [
    "race",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult"
]

for col in categorical_missing_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Unknown")

# Confirmar valores em falta restantes
missing_after = pd.DataFrame({
    "n_missing": df_clean.isna().sum(),
    "pct_missing": df_clean.isna().mean() * 100
}).sort_values("pct_missing", ascending=False)

missing_after[missing_after["n_missing"] > 0]

,n_missing,pct_missing


In [12]:
# Definir variável-alvo
target = "readmitted_30"

# Colunas que não devem entrar como variáveis explicativas
cols_to_exclude = [
    "encounter_id",   # identificador do episódio
    "patient_nbr",    # identificador do paciente
    "readmitted",     # variável-alvo original
    "readmitted_30"   # variável-alvo binária
]

# Criar X e y
X = df_clean.drop(columns=cols_to_exclude)
y = df_clean[target]

X.shape, y.shape

((101766, 44), (101766,))

In [13]:
# Identificar variáveis numéricas e categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Número de variáveis numéricas:", len(numeric_features))
print("Número de variáveis categóricas:", len(categorical_features))

print("\nVariáveis numéricas:")
print(numeric_features)

print("\nVariáveis categóricas:")
print(categorical_features)

Número de variáveis numéricas: 11
Número de variáveis categóricas: 33

Variáveis numéricas:
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Variáveis categóricas:
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


/var/folders/q8/07dvr7w958lgfy1598ltqybm0000gn/T/ipykernel_9460/1479702510.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [14]:
X.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,...,No,No,No,No,No,No,No,No,No,No
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,...,No,No,Up,No,No,No,No,No,Ch,Yes
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,...,No,No,No,No,No,No,No,No,No,Yes
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,...,No,No,Up,No,No,No,No,No,Ch,Yes
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,...,No,No,Steady,No,No,No,No,No,Ch,Yes


In [16]:
# Algumas variáveis são códigos categóricos, apesar de estarem como números
coded_categorical_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in coded_categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("object")

In [17]:
target = "readmitted_30"

cols_to_exclude = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30"
]

X = df_clean.drop(columns=cols_to_exclude)
y = df_clean[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

print("Número de variáveis numéricas:", len(numeric_features))
print("Número de variáveis categóricas:", len(categorical_features))

Dimensão de X: (101766, 44)
Dimensão de y: (101766,)
Número de variáveis numéricas: 8
Número de variáveis categóricas: 36


/var/folders/q8/07dvr7w958lgfy1598ltqybm0000gn/T/ipykernel_9460/3011695959.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nDistribuição da variável-alvo no treino:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribuição da variável-alvo no teste:")
print(y_test.value_counts(normalize=True) * 100)

X_train: (81412, 44)
X_test: (20354, 44)
y_train: (81412,)
y_test: (20354,)

Distribuição da variável-alvo no treino:
readmitted_30
0    88.839483
1    11.160517
Name: proportion, dtype: float64

Distribuição da variável-alvo no teste:
readmitted_30
0    88.842488
1    11.157512
Name: proportion, dtype: float64


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pré-processamento das variáveis numéricas
numeric_transformer = StandardScaler()

# Pré-processamento das variáveis categóricas
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Combinar os dois tipos de transformação
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [21]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("X_train original:", X_train.shape)
print("X_train preparado:", X_train_prepared.shape)

print("X_test original:", X_test.shape)
print("X_test preparado:", X_test_prepared.shape)


X_train original: (81412, 44)
X_train preparado: (81412, 2328)
X_test original: (20354, 44)
X_test preparado: (20354, 2328)


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Pipeline completo: pré-processamento + modelo
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Treinar o modelo
log_reg_model.fit(X_train, y_train)

# Previsões
y_pred_logreg = log_reg_model.predict(X_test)
y_proba_logreg = log_reg_model.predict_proba(X_test)[:, 1]

# Avaliação
print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_logreg))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_logreg))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_logreg))

Matriz de confusão:
[[12146  5937]
 [  969  1302]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.93      0.67      0.78     18083
           1       0.18      0.57      0.27      2271

    accuracy                           0.66     20354
   macro avg       0.55      0.62      0.53     20354
weighted avg       0.84      0.66      0.72     20354


AUC:
0.6699424029220123


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_rf))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_rf))

Matriz de confusão:
[[11423  6660]
 [  854  1417]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.93      0.63      0.75     18083
           1       0.18      0.62      0.27      2271

    accuracy                           0.63     20354
   macro avg       0.55      0.63      0.51     20354
weighted avg       0.85      0.63      0.70     20354


AUC:
0.6745345895496846


In [24]:
model_results = pd.DataFrame({
    "modelo": ["Regressão Logística", "Random Forest"],
    "accuracy": [0.66, 0.63],
    "precision_classe_1": [0.18, 0.18],
    "recall_classe_1": [0.57, 0.62],
    "f1_classe_1": [0.27, 0.27],
    "auc": [0.6699, 0.6745]
})

model_results

,modelo,accuracy,precision_classe_1,recall_classe_1,f1_classe_1,auc
0,Regressão Logística,0.66,0.18,0.57,0.27,0.6699
1,Random Forest,0.63,0.18,0.62,0.27,0.6745


In [25]:
model_results.to_csv("../outputs/tabelas/comparacao_modelos.csv", index=False)